In [1]:
from collections import defaultdict, Counter
import pandas as pd
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("Gunfish DATATATA - Base Data.csv", dtype={
    "P1 Fish": "category",
    "P1 Score": float,
    "P1 Rating": float,

    "P2 Fish": "category",
    "P2 Score": float,
    "P2 Rating": float,

    "P3 Fish": "category",
    "P3 Score": float,
    "P3 Rating": float,

    "P4 Fish": "category",
    "P4 Score": float,
    "P4 Rating": float,

    "Map": "category",
    "Avg Rating": float,
    "Player Count": int,
    "Avg Score": float,
})


In [3]:
df.columns = [col.strip() for col in df.columns]

In [4]:
fish_colors = {
    "Pufferfish": "gold",
    "Salmon": "salmon",
    "Swordfish": "steelblue",
    "Bass": "darkgreen",
    "Anglerfish": "orangered",
    "Needlefish": "cadetblue",
    "Flounder": "darkgoldenrod",
    "Eel": "dimgray",
    "Flyfish": "skyblue",
}

In [5]:
fish_df = pd.DataFrame({
    "Fish": df["P1 Fish"].to_list() + df["P2 Fish"].to_list() + df["P3 Fish"].to_list() + df["P4 Fish"].to_list(),
    "Score": df["P1 Score"].to_list() + df["P2 Score"].to_list() + df["P3 Score"].to_list() + df["P4 Score"].to_list(),
    "Rating": df["P1 Rating"].to_list() + df["P2 Rating"].to_list() + df["P3 Rating"].to_list() + df["P4 Rating"].to_list(),
})

In [6]:
def fish_bar(pdf, xcol, ycol, title, **kwargs):
    return pdf.plot.bar(xcol, ycol, title=title, color=[fish_colors[f] for f in pdf[xcol]], **kwargs)

In [7]:
score_cols = ["P1 Score", "P2 Score", "P3 Score", "P4 Score"]
win_cols = ["P1 Win", "P2 Win", "P3 Win", "P4 Win"]
fish_cols = ["P1 Fish", "P2 Fish", "P3 Fish", "P4 Fish"]

df["max_score"] = df[score_cols].max(axis=1)
df["full_stock"] = df["max_score"] == (df["Player Count"] - 1) * 3

scores = df[score_cols].copy()
for win_col, score_col in zip(win_cols, score_cols):
    scores[win_col] = scores[score_col] == df["max_score"]
scores["tie"] = scores[win_cols].sum(axis=1) > 1

for win_col, score_col in zip(win_cols, score_cols):
    scores[win_col] = scores[win_col] & (~scores.tie)

for fish_col in fish_cols:
    scores[fish_col] = df[fish_col]

fish_scores = pd.DataFrame({
    "Match_ID": df.index.to_list() + df.index.to_list() + df.index.to_list() + df.index.to_list(),
    "Fish": scores["P1 Fish"].to_list() + scores["P2 Fish"].to_list() + scores["P3 Fish"].to_list() + scores["P4 Fish"].to_list(),
    "Score": scores["P1 Score"].to_list() + scores["P2 Score"].to_list() + scores["P3 Score"].to_list() + scores["P4 Score"].to_list(),
    "Rating": df["P1 Rating"].to_list() + df["P2 Rating"].to_list() + df["P3 Rating"].to_list() + df["P4 Rating"].to_list(),
    "Win": scores["P1 Win"].to_list() + scores["P2 Win"].to_list() + scores["P3 Win"].to_list() + scores["P4 Win"].to_list(),
    "Map": df["Map"].to_list() + df["Map"].to_list() + df["Map"].to_list() + df["Map"].to_list()
}).sort_values(["Match_ID", "Win"])
fish_scores = fish_scores[~fish_scores.Score.isna()].reset_index(drop=True)

In [8]:
fish_map_appearances = (
    fish_scores
    .groupby(["Match_ID","Fish", "Map"])
    .count().reset_index()
    .groupby(["Fish", "Map"])
    .count().reset_index()
    .rename(columns={"Match_ID": "Appearances"})
    [["Fish", "Map", "Appearances"]]
)

In [9]:
fish_map_plays = (
    fish_scores
    [["Match_ID", "Fish", "Map"]]
    .groupby(["Fish", "Map"])
    .count().reset_index()
    .rename(columns={"Match_ID": "Plays"})
)

In [10]:
fish_map_wins = (
    fish_scores
    [["Fish", "Map", "Win"]]
    .groupby(["Fish", "Map"])
    .sum().reset_index()
    .merge(fish_map_appearances, on=["Fish", "Map"])
)
fish_map_wins["Win Rate"] = fish_map_wins["Win"] / fish_map_wins["Appearances"]
fish_map_wins.sort_values(["Map", "Win Rate"], inplace=True)

In [11]:
map_win_piv = fish_map_wins.pivot(index="Fish", columns="Map", values="Win Rate").reset_index()
map_win_piv.columns = [f"{col}_win_rate" if col != "Fish" else col for col in map_win_piv.columns]

In [12]:
non_score_by_map = (
    fish_scores[fish_scores["Score"] == 0]
    [["Fish", "Map"]]
    .groupby("Map")
    .count().reset_index()
    .rename(columns={"Fish": "Non-Scorers"})
)

In [13]:
non_score_by_fish_by_map = (
    fish_scores[fish_scores["Score"] == 0]
    [["Fish", "Win", "Map"]]
    .groupby(["Map", "Fish"])
    .count().reset_index()
    .rename(columns={"Win": "Non-Scorers"})
    .sort_values(["Map", "Non-Scorers"])
)

In [14]:
non_score_by_fish_by_map_by_plays = non_score_by_fish_by_map.merge(fish_map_plays, on=["Fish", "Map"])
non_score_by_fish_by_map_by_plays["Non-Score Rate"] = (
    non_score_by_fish_by_map_by_plays["Non-Scorers"]
    / non_score_by_fish_by_map_by_plays["Plays"]
)

In [15]:
non_score_piv = non_score_by_fish_by_map.pivot(index="Fish", columns="Map", values="Non-Scorers").reset_index()
non_score_piv.columns = [f"{col}_non_scorers" if col != "Fish" else col for col in non_score_piv.columns]

In [16]:
negative_ratings_by_fish_by_map = (
    fish_scores
    [fish_scores["Rating"] == -1]
    [["Fish", "Map", "Rating"]]
    .groupby(["Map", "Fish"])
    .count().reset_index()
    .rename(columns={"Rating": "Negative Ratings"})
    .sort_values(["Map", "Negative Ratings"])
)

In [17]:
pplays = (~scores[score_cols].isna()).sum().reset_index()
pplays.columns = ["Player", "Plays"]
pplays["Player"] = pplays["Player"].str.strip(" Score")

pwins = scores[win_cols].sum().reset_index()
pwins.columns = ["Player", "Wins"]
pwins["Player"] = pwins["Player"].str.strip(" Win")
pwins = pwins.merge(pplays, on="Player")
pwins["wnorm"] = pwins["Wins"] / pwins["Plays"]

In [18]:
fish_plays = fish_scores[["Fish", "Win"]].groupby("Fish").count().reset_index().rename(columns={"Win": "Plays"}).sort_values("Plays")

In [19]:
fishes = [f for f in fish_df["Fish"].unique() if not pd.isna(f)]

fish_apps = pd.DataFrame()
fish_homos = pd.DataFrame()

for fish in fishes:
    fish_apps[fish] = (df[fish_cols] == fish).any(axis=1)
    fish_homos[fish] = ((df[fish_cols] == fish) | df[fish_cols].isna()).all(axis=1)

fish_apps = fish_apps.sum().reset_index()
fish_apps.columns = ["Fish", "Appearances"]

fish_homos = fish_homos.sum().reset_index()
fish_homos.columns = ["Fish", "Homogeneous Matches"]
fish_homos.sort_values("Homogeneous Matches", inplace=True)

In [20]:
fish_scores["Scoreless"] = fish_scores["Score"] == 0
scoreless = fish_scores[["Fish", "Scoreless"]].groupby("Fish").sum().reset_index().sort_values("Scoreless").merge(fish_plays, on="Fish")
scoreless["Scoreless Percent"] = scoreless["Scoreless"] / scoreless["Plays"]
scoreless.sort_values("Scoreless Percent", inplace=True)

In [21]:
negative_ratings_per_fish_appearance = (
    fish_scores
    [fish_scores["Rating"] == -1]
    [["Fish", "Rating"]]
    .groupby("Fish")
    .count().reset_index()
    .merge(fish_apps, on="Fish")
    .rename(columns={"Rating": "Negative Ratings"})
)
negative_ratings_per_fish_appearance["Negative Ratings per Appearance"] = negative_ratings_per_fish_appearance["Negative Ratings"] / negative_ratings_per_fish_appearance["Appearances"]
negative_ratings_per_fish_appearance.sort_values("Negative Ratings per Appearance", inplace=True)

In [22]:
duos = defaultdict(list)
matchups = list()

for row in df[df["Player Count"] == 2].itertuples():
    p1_found = False
    for col in fish_cols:
        cidx = df.columns.get_loc(col) + 1
        if not (pd.isna(row[cidx + 1]) or p1_found):
            f1 = row[cidx]
            s1 = row[cidx + 1]
            r1 = row[cidx + 2]
            p1_found = True
        elif not pd.isna(row[cidx + 1]):
            f2 = row[cidx]
            s2 = row[cidx + 1]
            r2 = row[cidx + 2]

    # winner stats
    duos["wf"].append(f1 if s1 >= s2 else f2)
    duos["ws"].append(s1 if s1 >= s2 else s2)
    duos["wr"].append(r1 if s1 >= s2 else r2)

    # loser stats
    duos["lf"].append(f1 if s1 < s2 else f2)
    duos["ls"].append(s1 if s1 < s2 else s2)
    duos["lr"].append(r1 if s1 < s2 else r2)

    # others
    duos["m"].append(row.Map)
    duos["t"].append(s1 == s2)

    # matchups
    duos["m1"].append(f1 if f1 <= f2 else f2)
    duos["m2"].append(f2 if f1 <= f2 else f1)

    matchups.append((f1 if f1 <= f2 else f2,
                     f2 if f1 <= f2 else f1))

matchups = Counter(matchups)
duos = pd.DataFrame(duos)
duos["sdiff"] = duos.ws - duos.ls

In [23]:
vs_match = (
    duos[~duos.t][["m1", "m2", "m"]]
    .groupby(["m1", "m2"])
    .count().reset_index()
    .rename(columns={"m": "match"})
)

In [24]:
vs_wins = (
    duos[~duos.t][["wf", "lf", "m"]]
    .groupby(["wf", "lf"])
    .count().reset_index()
    .rename(columns={"m": "wins"})
)
vs_wins["m1"] = vs_wins.apply(lambda row: row.wf if row.wf <= row.lf else row.lf, axis=1)
vs_wins["m2"] = vs_wins.apply(lambda row: row.lf if row.wf <= row.lf else row.wf, axis=1)

vs_wins = vs_wins.merge(vs_match, on=["m1", "m2"])

vs_wins["Win Rate"] = vs_wins["wins"] / vs_wins["match"]
# vs_wins = vs_wins.drop(columns=["m1", "m2", "wins", "match"]).rename(columns={"wf": "Winning Fish", "lf": "Losing Fish"})

In [25]:
vs_fish_wins = (
    duos[~duos.t]
    [["wf", "m"]]
    .groupby("wf")
    .count().reset_index()
    .rename(columns={"wf": "Fish", "m": "Wins"})
)

In [26]:
vs_fish_loss = (
    duos[~duos.t]
    [["lf", "m"]]
    .groupby("lf")
    .count().reset_index()
    .rename(columns={"lf": "Fish", "m": "Losses"})
)

In [27]:
vs_fish_stats = vs_fish_wins.merge(vs_fish_loss, on="Fish")
vs_fish_stats["Appearances"] = vs_fish_stats.Wins + vs_fish_stats.Losses
vs_fish_stats["Vs Win Rate"] = vs_fish_stats.Wins / vs_fish_stats.Appearances

In [28]:
mean_score_diff_piv = (
    duos
    [["wf", "lf", "sdiff"]]
    .groupby(["wf", "lf"])
    .mean().reset_index()
    .rename(columns={"wf": "Fish"})
    .pivot(index="Fish", columns="lf", values="sdiff")
    .reset_index()
)
mean_score_diff_piv.columns = [f"{col.lower()}_sdiff" if col != "Fish" else col for col in mean_score_diff_piv.columns]

In [29]:
fish_metrics = map_win_piv.merge(
    non_score_piv, on="Fish"
).merge(
    mean_score_diff_piv, on="Fish"
).set_index("Fish").fillna(0)


In [30]:
fish_similarities = defaultdict(list)

for f1 in fishes:
    for f2 in fishes:
        sim = cosine_similarity(
            fish_metrics.loc[f1].values.reshape(1, -1),
            fish_metrics.loc[f2].values.reshape(1, -1)
        )[0][0]
        fish_similarities["f1"].append(f1)
        fish_similarities["f2"].append(f2)
        fish_similarities["sim"].append(sim)

fish_similarities = pd.DataFrame(fish_similarities).pivot(index="f1", columns="f2", values="sim")

# GunFish Stats!
Data analysis for GunFish in MAGFest's Indie Arcade 2025.

## Basics
Sanity check fundamentals, like whether one player-controller wins more than others.
Display baseline stats like play count, number of non-scoring players, etc.

Initial takeaway is that somewhat less than a third of players get **no kills at all**.  
Likewise, no player in the entire convention managed to stock-out all of their foes in a 4-player game single-handedly,
and only **once** did a player manage to single-handedly stock-out both opponents in a 3-player match.  

In [ ]:
pd.DataFrame({
    "Total Games": [len(df)],
    "Total Points": [fish_scores["Score"].sum()],
    "Total Players": [(~fish_scores["Score"].isna()).sum()],
    "Non-Scorers": [(fish_scores["Score"] == 0).sum()],
    "Scoring Percent": [f'{(fish_scores["Score"] > 0).sum() / (~fish_scores["Score"].isna()).sum():0.0%}'],
    "Highest Score": [fish_scores["Score"].max()],  # nobody got all nine stocks!
    "Many-Player Stock-Outs": [df[df["Player Count"] > 2]["full_stock"].sum()],
})

The game where a player managed to take all the stocks of both opponents was a Pufferfish dominating two Eels on the Beach.  
The Pufferfish rated the game positively, and the others did not rate. It is altogether possible this was one person at
the cabinet playing solo - something I saw a handful of times.  

In [ ]:
df[(df["Player Count"] > 2) & df["full_stock"]]

There is a slight trend towards P2 victories. Might not be significant?

In [ ]:
n = pwins[["wnorm"]]
n.index = pwins["Player"]
n.plot.pie(y="wnorm", title="Player Wins per Play", figsize=(6, 6), colors=["red", "green", "mediumblue", "gold"])

1-v-1 matches make up **more than 50%** of all matches! This gameplay should get extra attention in map deisgn and playtesting.  

In [ ]:
((df["Player Count"].value_counts() / len(df)) * 100).plot(kind="pie", title="Player Counts")

## Fish Stats
Stat breakdowns by Fish. Popularity, average score, etc.

### Fish Popularity
The **Bassic Fish** is the most frequently-chosen. This is expected, as it is the default.  
Flyfish, Salmon, and Pufferfish are the next three fish to the right of the Bassic,
which may explain their prevalence.  
Anglerfish has a clear, appealing concept from the art, while Needlefish is rendered somewhat small,
potentially explaining their positions.  

In [ ]:
pop = fish_df.Fish.value_counts().reset_index().rename(columns={"count": "Times Chosen"}).sort_values("Times Chosen")
fish_bar(pop, "Fish", "Times Chosen", "Fish Popularity")

### Homogeneous Matches
By far, most games that were played with all the same fish were played with the **Bassic Fish**.  
This is, again, not unexpected as the Bassic Fish is the default.  
Swordfish appearing relatively often in homogeneous matches is perhaps interesting -
likely due to its power and control. Do we feel that the Swordfish should be made less
appealing in our game about being fishes with guns in their mouths?  
Perhaps more interesting is the Flyfish being second-most-homogeneous, despite being, statistically, "mid".
Perhaps there are players recalling its dominance from last year?  

In [ ]:
fish_bar(fish_homos, "Fish", "Homogeneous Matches", "Matches with All the Same Fish")

### Average Score per Play
This examines average score per fish per *play* -
this means that if a match is played between 4 Bass, that is 4 plays for the Bass.

Salmon, Swordfish, and Pufferfish having the highest average score is not unexpected.
They are powerful fish with either high damage potential or a great deal of control.  

Worthy of note is the Anglerfish and Bass averaging **less than 1 point per appearance**.
Ideally, we should produce few matches where players do not manage a single point.  

In [ ]:
kpapp = fish_df[["Fish", "Score"]].groupby("Fish").mean().reset_index().rename(columns={"Score": "Average Score"}).sort_values("Average Score")
fish_bar(kpapp, "Fish", "Average Score", "Average Score per Play")

### Non-Scoring Plays
Given that there are fish averaging less than 1 point per play, how many total non-scoring
play-sessions did each fish have?  

Before we begin, there is a major confounding factor in this data that should give us pause -
sometimes players were seen to start matches with multiple fish and only one player, as they
proceeded to practice picking off the AFK fish. These sessions would produce non-scoring plays
for the fish chosen, and may contribute to the Bassic Fish's second-place status here, it being
the default fish.  

That said, the Anglerfish fails to score nearly *half* of the time, demonstrating that it needs
adjustment either in its damage output or in its controllability. Likewise the Eel seems to
suffer - likely with damage output in maps where there is insufficient water to use its mechanical
bonuses. These results seem reliable, as the fish in question are not the default Bassic fish,
and likely represent genuine play-sessions.  

The Bassic Fish also running about a 40% non-scoring rate seems undesirable. Sure, new players
may require a play session or two in order to become familiar with the controls, but making it
easier to score would likely increase the likelihood of their putting in that session-or-two.
Lacking data from which to estimate how many players put in a single session and then walked
away, however, leaves us only with a hunch in this respect. It is possible that most players
put in the session-or-two necessary to figuring out the controls regardless of whether they
have a non-scoring first session.  

In [ ]:
nspf = fish_df[fish_df["Score"] == 0][["Fish", "Rating"]].groupby("Fish").count().reset_index().rename(columns={"Rating": "Non-Scoring Plays"})
tplays = fish_df[["Fish", "Rating"]].groupby("Fish").count().reset_index().rename(columns={"Rating": "Total Plays"})
nspf = nspf.merge(tplays, on="Fish")
nspf["Non-Scoring Plays per Play"] = nspf["Non-Scoring Plays"] / nspf["Total Plays"]
nspf.sort_values("Non-Scoring Plays per Play", inplace=True)
fish_bar(nspf, "Fish", "Non-Scoring Plays per Play", "Non-Scoring Plays per Play per Fish")

### Fish Wins per Appearance
This counts times when a fish got a non-tying high score divided by matches
where *any* player chose that fish.

These results are very similar to the Average Score per Play results,
except that the Bass performs better while the Eel and Flounder drop. This makes
sense, as the many all-Bass games guarantee Bass wins whenever they occur. The
Salmon's dominant performance, winning nearly *half* of all matches where a
Salmon is chosen, with very few Salmon-only matches, demonstrates strongly that
"GunFish devs pls nerf Salmon" is a respectable and correct sentiment.  

In [ ]:
fwins = fish_scores[["Fish", "Win"]].groupby("Fish").sum().reset_index().merge(fish_apps, on="Fish")
fwins["Wins per Appearance"] = fwins["Win"] / fwins["Appearances"]
fwins.sort_values("Wins per Appearance", inplace=True)
fish_bar(fwins, "Fish", "Wins per Appearance", "Fish Wins per Appearance")

### Fish Win Rate in 1-v-1s
The 1-v-1 Win Rate for fish is very similar to the win rate in non-1-v-1s.

In [ ]:
fish_bar(vs_fish_stats.sort_values("Vs Win Rate"), "Fish", "Vs Win Rate", "Fish Win Rate in 1-v-1s")

### Negative Ratings per Fish Appearance
Setting aside that the data set is too small to reach any solid conclusions, this is what we can find from negative ratings.  

The Bass and Eel appearing near the top of this chart is unsurprising, as Bass is likely to be played by
a new player who has not (or will not) figure out the controls. The Eel simply does not do very well in
most games. The Pufferfish being third is more surprising, as it typically performs quite well in terms of
victory and scoring metrics. This perhaps is due to the Pufferfish's control scheme being one of the less
intuitive of the "special fish" (two button-presses in order to detonate the grenade early). A player might also
perform well as the Pufferfish, but recognize that this performance is "bullshit" and due to an unbalanced
situation.  

The Swordfish producing the fewest negative reviews is likely due to it being the easiest fish to control, and
it performing well mechanically. This again raises the question of whether we want the Fish without a Gun to be
so good and fun in GunFish. Naturally, making the game less fun overall by removing the Swordfish seems like the
wrong response, so perhaps the correct line of action is *not* to nerf the Salmon or Swordfish, but rather to
*buff* all other fish? Naturally, the actual balance implications of doing so are difficult to predict, but if
players are having fun doing a lot of damage or having high control, maybe we should find ways to bring that
gameplay to more fish.  

In [ ]:
fish_bar(negative_ratings_per_fish_appearance, "Fish", "Negative Ratings per Appearance", "Negative Ratings per Fish Appearance")

Correlation between fish Rating and Score. There does not appear to be a strong link between the two.

In [ ]:
fish_scores[fish_scores["Rating"] != 0]["Rating"].corr(fish_scores["Score"])

Likewise, there does not appear to be a strong link between Rating and victory in the match.

In [ ]:
fish_scores[fish_scores["Rating"] != 0]["Rating"].corr(fish_scores["Win"])

## Map Stats
Breakdowns of stats by Map to identify trends between fish performance and map nature. Infer connections to
number of map hazards, sight-lines, etc.  

### Average Score per Map
Barrel is the highest scoring map, which is not unexpected, as it places all the
fish within easy reach of one-another, without any map hazards.  
Likewise, Valley, Acid Factory, and Firing Range being lower-scoring is explicable
due to their broken-up sightlines, and deadly map hazards.  

In [ ]:
avg_score_map = (
    fish_scores
    [["Map", "Score"]]
    .groupby("Map")
    .mean().reset_index()
    .rename(columns={"Score": "Average Score"})
    .sort_values("Average Score")
)
avg_score_map.plot.bar("Map", "Average Score", title="Average Score per Map")

### Average Score per Fish per Map
How does this break out per Fish?

We can see a clear impact where the raycast fish like the Bass and Needlefish struggle
more on maps with more restricted sightlines, such as Cargo Hold, Valley, or Acid Factory.
In that vein, the Flounder appears to be underperforming on such maps, despite being another
raycast-centric fish, though it does have a fairly short range. Likely, it needs a small buff.  

Finally, if we want to allow the **Bassic Fish** to be more viable on most maps, we
should consider having maps be somewhat less tight, which may improve the game experience
for new players choosing the default fish. At the same time, maps with a lot going
on are chaotic and fun, at least in the opinion of this dev.  

In [ ]:
mkpapp = (
    fish_scores[["Fish", "Map", "Score"]]
    .groupby(["Map", "Fish"])
    .mean().reset_index()
    .rename(columns={"Score": "Average Score"})
    .sort_values(["Map", "Average Score"])
)
ax = sns.catplot(x="Map", y="Average Score", hue="Fish", data=mkpapp, kind="bar", palette=fish_colors, height=5, aspect=2)
ax.set(title="Average Score per Fish per Map")

### Win Rate per Fish per Map


In [ ]:
ax = sns.catplot(x="Map", y="Win Rate", hue="Fish", data=fish_map_wins, kind="bar", palette=fish_colors, height=5, aspect=2)
ax.set(title="Win Rate per Fish per Map")

### Rating Rate
To test that hypothesis of "chaos -> fun", we look at the ratings players gave to matches on
given maps, broken out by fish. First, however, we must consider how many matches were actually rated.  

In [ ]:
f"Out of {len(fish_scores)} plays, we received {(fish_scores.Rating != 0).sum()} non-zero Ratings, for a {(fish_scores.Rating != 0).sum() / len(fish_scores):0.0%} rating - uh - rate."

### Average Rating per Map
Given the caveat that only 1/3 players rated the game, we can start looking at the breakdown
by map, and then by fish and map.  

The first takeaway is that, even though Acid Factory is one of the lowest maps in terms of
average score, it nevertheless is rated highly. If we hazard to draw conclusions from this scant
data, we could say that Acid Factory is just plain-old fun. This, actually, reflects the devs'
sentiments from playtesting - Acid Factory was one of our favorites, too.  

Valley and Firing Range do not seem to make up for their low average scores with good ratings,
so likely they need some tuning in order to reach the heights of Acid Factory.  

In [ ]:
map_ratings = (
    fish_scores
    [fish_scores["Rating"] != 0]
    [["Map", "Rating"]]
    .groupby("Map")
    .mean().reset_index()
    .rename(columns={"Rating": "Average Rating"})
    .sort_values(["Map", "Average Rating"])
)
map_ratings.plot.bar("Map", "Average Rating")

### Average Rating per Map per Fish
Breaking down rating by fish reveals that the Flounder and Needlefish rated some maps
highly despite their lackluster performance. They were both solidly middle-of-the-road
in terms of mechanical performance on Acid Factory, but gave it universally positive ratings.  

Likely, however, we should not draw too many conclusions from this data, given its sparse
nature.

In [ ]:
mrpapp = (
    fish_scores
    [fish_scores["Rating"] != 0]
    [["Fish", "Map", "Rating"]]
    .groupby(["Fish", "Map"])
    .mean().reset_index()
    .rename(columns={"Rating": "Average Rating"})
    .sort_values(["Map", "Average Rating"])
)
ax = sns.catplot(x="Map", y="Average Rating", hue="Fish", data=mrpapp, kind="bar", palette=fish_colors, height=5, aspect=2)
ax.set(title="Average Rating per Fish per Map")

### Negative Rating Count per Fish per Map
Again, these counts are small enough that they *could* be due to simple
**gamer variation** - e.g. we saw some individuals playing many games in a row,
and rating negatively each time. Presumably these negative ratings were not
*quite* reflective of their true sentiment, but so it goes.  

If we attempt to look past that, however, we can see that the Bassic Fish was
likely handing out more negative ratings due to being the choice for new players,
though the maps with open sight-lines perform meaningfully better even in this case.  

In [ ]:
ax = sns.catplot(x="Map", y="Negative Ratings", hue="Fish", data=negative_ratings_by_fish_by_map, kind="bar", palette=fish_colors, height=5, aspect=2)
ax.set(title="Negative Rating Count per Fish per Map")

### Non-Scoring Rate per Fish per Map
Looking at the number of non-scoring plays per fish per map, we can see the results re-emphasized
that Acid Factory gets similar results in terms of low-scoring games, but nevertheless is fun
enough that players don't *seem* to disprefer it to other maps (at least not too strongly).

In [ ]:
ax = sns.catplot(x="Map", y="Non-Score Rate", hue="Fish", data=non_score_by_fish_by_map_by_plays, kind="bar", palette=fish_colors, height=5, aspect=2)
ax.set(title="Non-Scoring Rate per Fish per Map")

## 1-v-1 Stats

### Win Rate in 1-v-1s Between Fishes
From these matchups, it is clear that the Swordfish outperforms many other fish,
while the likes of the Flounder generally underperform. The Bass appears to perform relatively poorly in head-to-head matches, as well.  

In [ ]:
vs_wins_piv = vs_wins.rename(columns={"wf": "Winning Fish", "lf": "Losing Fish"}).pivot(index="Winning Fish", columns="Losing Fish", values="Win Rate")
ax = sns.heatmap(vs_wins_piv, annot=True, fmt="0.0%")
ax.set(title="Win Rate in 1-v-1s Between Fishes")

### Average Winning Score Differential in 1-v-1s Between Fishes


In [ ]:
sdiffs = (
    duos[~duos.t][["wf", "lf", "sdiff"]]
    .groupby(["wf", "lf"])
    .mean().reset_index()
    .rename(columns={"sdiff": "Average Winning Score Differential", "wf": "Winning Fish", "lf": "Losing Fish"})
)
sdiffs_piv = sdiffs.pivot(index="Winning Fish", columns="Losing Fish", values="Average Winning Score Differential")
ax = sns.heatmap(sdiffs_piv, annot=True)
ax.set(title="Average Winning Score Differential in 1-v-1s Between Fishes")

## Fish Similarity!
Metrical similarity between fishes, per the cosine similarity between fish in these metrics:

In [ ]:
list(fish_metrics.columns)

The high-similarity parings of the Bass with the Needlefish and the Eel with the Pufferfish make sense given that the former are our bog-standard raycast-em-once fish and the latter are our lob-a-GameObject-with-AOE-at-em fish. They'll be similarly advantaged or disadvantaged by similar map layouts, objects, lineups, etc.  

The Flyfish's relative dissimilarity from the Salmon, despite their both being "machine gun fishes", is likely due to the latter's
dominant performance thoroughly overshadowing the Flyfish's statistically "mid" performance.

In [ ]:
ax = sns.heatmap(fish_similarities, annot=True)
ax.set(title="Fish Cosine Similarity")

# Conclusions & Additional Thoughts
## Fish Balance
The clear and decisive outlier performance of the Salmon recommends adjustments to bring it closer to baseline.
The best way to do this, however, may be to improve the performance of the other fish, rather than to nerf the
Salmon. Conversely, the Pufferfish seems like it needs a reduction in performance, or at least an adjustment
that takes into account the fact that players seemed to rate it poorly, despite its good mechanical performance.  

Giving the Pufferfish a smaller radius for its AOE, but with less sharp damage dropoff, may be an appropriate
initial adjustment, but improving the other fish to come closer to the Salmon likely requires more fundamental
adjustments to how the game controls. Possibly, giving player more control over the rotation of their fish would
achieve this end, though the Salmon may also deserve a minor damage debuff in addition, especially if it is going
to benefit from increased accuracy granted by better fish control.  

Apart from the high-performing fish, the Flounder and Anglerfish could do with some help - likely a quicker buildup
of charge & damage for the Anglerfish, with either higher damage, or perhaps even a tighter spread on raycasts for
the Flounder. If a means of better fish control can be found, though, these fish may already find themselves to be
sufficiently adjusted. It also should be seen whether the Anglerfish might simply perform adequately in more-
experienced hands. Likely, a dev-only stats gathering session is in order, so that we can see what performance looks
like when fish are exclusively controlled by players who know the game mechanics well.  

Finally, maybe this is all wrong! Maybe we're drawing overly-strong conclusions from insufficient data! Maybe the
game is already fun enough! I love data nalysis.  

## Map Balance
There *appears* to be a distinction between what I will call "open maps" and "closed maps". The "closed maps" seem to produce
lower scores and lower ratings, apart from Acid Factory, which is just **fun**. This does not necessarily mean that our "closed
maps" (e.g. Valley or Firing Range) need to be made "open", just that there may be additions necessary to make them more fun.
One feature of Acid Factory that may influence this trend is that it has a lot of moving pieces. Sure, there are the large wheel,
the acid pits, and the map objects around to break up sight lines, but there are also many ways to move up and around these map
features (the moving platform, the canon, and indeed the wheel itself).  

Something that *absolutely* should figure into our future map design decisions is that **most games are 1-v-1**! We need to
investigate all kinds of things with that in mind - spawn point distribution, map traversal time, hazard density - the works.  


## Future Data Collection

### Solo Play
We should try to differentiate between actual games and "solo play", where an individual activates multiple players but is just practicing shooting AFK fish. I saw this happen a few times, and it perhaps could be detected by logging input counts per fish so that we can discard examples that have few enough inputs to be considered "AFK".  

### Timestamps
We should log match duration, or time of start and time of stop. This can help us figure out when there are "runs" of matches and possibly identify individual play sessions by the same players.  

Likewise, we should log player deaths individually, with timestamps, so that we can identify the longest-lived fish, etc..

### Harm Stats
We could also start logging individual instances of damage to a fish, both from map hazards and from fish shots, so that we can
begin to differentiate between deaths due to hazards and deaths due to fish. Likewise, we could begin identifying which fish put
out a large amount of damage, but don't manage to push their opponents over the "finish line" in order to score a kill.  

### Draft Schema
To implement these ideas, we should consider separating our logs into "fish logs" and "match logs" with keys between them.  
A tall, transactional log per fish life would be more useful than the current format. Player (P1, P2, P3, P4) should be a column. Likewise, we can log individual fish stats in a table, and then match stats in another table.  

The idea is to have things like:
* `fish_lives`: log match_id, player_id, cause_of_death, life_start_dt, life_end_dt, etc.
* `fishes`: log match_id, player_id, kills, score, etc.
* `matches`: log match_id, map, match_start_dt, match_stop_dt, etc.
* `fish_harm`: log match_id, player_id, damage_source, damage_quantity, is_killing_blow, etc.